# SEC EDGAR Financial Filings Collection

Collect company filings from the U.S. Securities and Exchange Commission (SEC) EDGAR system and convert them into provenance-preserving `EvidenceRecord` objects for the AAI-520 research workflow. The notebook uses the same Cohort 1 `NotebookResearchProvider` used by the research graph.

**Responsible use:** provide a descriptive `SEC_USER_AGENT` with a contact address, use reasonable request rates, and check filing content at the source URL before relying on it.

## 1. Initialize the project and configuration

Run from the repository root or from the `notebooks/` directory. The notebook reads `STOCK_TICKER` and `SEC_USER_AGENT` from environment variables when available.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
os.chdir('/content/drive/MyDrive/AAI-520-Project')

print(f"current dir: {Path.cwd()}")

Mounted at /content/drive
current dir: /content/drive/MyDrive/AAI-520-Project


In [3]:
import os
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
if not (project_root / "src").exists():
    raise FileNotFoundError("Could not locate the project src/ directory. Run from the repository or notebooks directory.")

sys.path.insert(0, str(project_root / "src"))

stock_ticker = os.getenv("STOCK_TICKER", "AAPL").strip().upper()
sec_user_agent = os.getenv("SEC_USER_AGENT", "AAI520 Investment Research Agent contact: research@example.com")
max_filings = int(os.getenv("SEC_MAX_FILINGS", "8"))

print(f"Project root: {project_root}")
print(f"Ticker: {stock_ticker}")
print(f"Maximum filings: {max_filings}")
print("Set SEC_USER_AGENT to your application's name and a real contact address before using SEC endpoints.")

Project root: /content/drive/MyDrive/AAI-520-Project
Ticker: AAPL
Maximum filings: 8
Set SEC_USER_AGENT to your application's name and a real contact address before using SEC endpoints.


## 2. Fetch SEC EDGAR filings

The adapter resolves the ticker to a Central Index Key (CIK), requests the SEC submissions index, and downloads primary documents for recent annual, quarterly, current, and selected foreign-issuer filings. Query terms can optionally restrict the form types (for example, `10-K` or `10-Q`).

In [4]:
from tools.notebook_adapter import NotebookResearchProvider

provider = NotebookResearchProvider(
    ticker=stock_ticker,
    max_filings=max_filings,
    sec_user_agent=sec_user_agent,
)

filing_query = "Recent annual, quarterly, and current financial filings"
filings = provider.filings(ticker=stock_ticker, query=filing_query)
print(f"Collected {len(filings)} SEC filing evidence records for {stock_ticker}.")

Collected 8 SEC filing evidence records for AAPL.


## 3. Inspect the collected evidence

Each record preserves the SEC source, filing title, filing date, ticker, canonical filing URL, accession number, and a bounded plain-text excerpt of the primary document.

In [5]:
import pandas as pd

filings_df = pd.DataFrame(
    [
        {
            "ticker": item.ticker,
            "source": item.source,
            "title": item.title,
            "published_at": item.published_at,
            "document_id": item.document_id,
            "url": item.url,
            "content_characters": len(item.content),
            "content_preview": item.content[:500],
        }
        for item in filings
    ]
)

if filings_df.empty:
    print("No filings returned. Verify the ticker/CIK, SEC_USER_AGENT, network access, and query form filters.")
else:
    display(filings_df[["ticker", "title", "published_at", "document_id", "url", "content_characters"]])
    display(filings_df[["title", "content_preview"]])

,ticker,title,published_at,document_id,url,content_characters
0,AAPL,Apple Inc. 10-Q filing filed 2026-07-31 — 10-Q,2026-07-31,0000320193-26-000020,https://www.sec.gov/Archives/edgar/data/320193...,20000
1,AAPL,Apple Inc. 8-K filing filed 2026-07-30 — 8-K,2026-07-30,0000320193-26-000018,https://www.sec.gov/Archives/edgar/data/320193...,3922
2,AAPL,Apple Inc. 10-Q filing filed 2026-05-01 — 10-Q,2026-05-01,0000320193-26-000013,https://www.sec.gov/Archives/edgar/data/320193...,20000
3,AAPL,Apple Inc. 8-K filing filed 2026-04-30 — 8-K,2026-04-30,0000320193-26-000011,https://www.sec.gov/Archives/edgar/data/320193...,3928
4,AAPL,Apple Inc. 8-K filing filed 2026-04-20 — 8-K,2026-04-20,0001140361-26-015711,https://www.sec.gov/Archives/edgar/data/320193...,4864
5,AAPL,Apple Inc. 8-K filing filed 2026-02-24 — 8-K,2026-02-24,0001140361-26-006577,https://www.sec.gov/Archives/edgar/data/320193...,5457
6,AAPL,Apple Inc. 10-Q filing filed 2026-01-30 — 10-Q,2026-01-30,0000320193-26-000006,https://www.sec.gov/Archives/edgar/data/320193...,20000
7,AAPL,Apple Inc. 8-K filing filed 2026-01-29 — 8-K,2026-01-29,0000320193-26-000005,https://www.sec.gov/Archives/edgar/data/320193...,3938


,title,content_preview
0,Apple Inc. 10-Q filing filed 2026-07-31 — 10-Q,aapl-20260627 false2026Q30000320193--09-26P1YP...
1,Apple Inc. 8-K filing filed 2026-07-30 — 8-K,aapl-20260730 false000032019300003201932026-07...
2,Apple Inc. 10-Q filing filed 2026-05-01 — 10-Q,aapl-20260328 false2026Q20000320193--09-26P1YP...
3,Apple Inc. 8-K filing filed 2026-04-30 — 8-K,aapl-20260430 false000032019300003201932026-04...
4,Apple Inc. 8-K filing filed 2026-04-20 — 8-K,truetruetruetruetruetrueNASDAQNASDAQNASDAQNASD...
5,Apple Inc. 8-K filing filed 2026-02-24 — 8-K,truetruetruetruetruetrueNASDAQNASDAQNASDAQNASD...
6,Apple Inc. 10-Q filing filed 2026-01-30 — 10-Q,aapl-20251227 false2026Q10000320193--09-26P1YP...
7,Apple Inc. 8-K filing filed 2026-01-29 — 8-K,aapl-20260129 false000032019300003201932026-01...


## 4. Save a local snapshot

The JSON snapshot can be reused for offline experiments. It contains public filing text and source metadata; review local data-handling requirements before committing it to version control.

In [6]:
from datetime import datetime, timezone
import json

output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"filings_{stock_ticker}.json"

payload = {
    "ticker": stock_ticker,
    "collected_at": datetime.now(timezone.utc).isoformat(),
    "source": "SEC EDGAR",
    "filings": [item.model_dump() for item in filings],
}
output_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved {len(filings)} filing records to {output_path}")

Saved 8 filing records to /content/drive/MyDrive/AAI-520-Project/data/processed/filings_AAPL.json


## 5. Handoff to the research workflow

Pass the `filings` list directly to `NotebookResearchProvider` or to the Cohort 1 provider boundary. The research graph can then include these SEC records in its evidence set and preserve their URLs for citations.

```python
from tools.notebook_adapter import NotebookResearchProvider

provider = NotebookResearchProvider(
    ticker=stock_ticker,
    filings=filings,  # optional offline snapshot override
)
assert all(item.source == "SEC EDGAR" for item in provider.filings(stock_ticker, ""))
```

The adapter's explicit `filings=` constructor argument remains useful for deterministic offline runs and tests; when no records are supplied, `filings()` fetches from SEC EDGAR.